In [4]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Refactored Functions

In [5]:
import pandas as pd


def file_loader(file_path):
    """Load a CSV file into a pandas dataframe."""
    return pd.read_csv(file_path)


def remove_blank_rows(df):
    """Remove rows where every column is blank."""
    return df.dropna(how="all").copy()


def duplicate_check(df):
    """Return duplicate row count."""
    return df.duplicated().sum()


def na_check(df):
    """Return missing value counts by column."""
    return df.isna().sum().to_frame("missing_count")


def data_enrich(df, checkout_col, returned_col):
    """
    Calculate the number of days between checkout and returned dates.
    Adds the result as a new column called days_borrowed.
    """
    df = df.copy()
    df["days_borrowed"] = (df[returned_col] - df[checkout_col]).dt.days
    return df


def clean_books_data(df):
    """Clean the books dataframe and set correct data types."""
    df = remove_blank_rows(df)

    df["Books"] = df["Books"].astype("string").str.strip()
    df["Days allowed to borrow"] = df["Days allowed to borrow"].astype("string").str.strip()

    df["Id"] = pd.to_numeric(df["Id"], errors="coerce").astype("Int64")
    df["Customer ID"] = pd.to_numeric(df["Customer ID"], errors="coerce").astype("Int64")

    df["Book checkout"] = (
        df["Book checkout"]
        .astype("string")
        .str.replace('"', '', regex=False)
        .str.strip()
    )

    df["Book Returned"] = (
        df["Book Returned"]
        .astype("string")
        .str.replace('"', '', regex=False)
        .str.strip()
    )

    df["Book checkout"] = pd.to_datetime(
        df["Book checkout"],
        errors="coerce",
        dayfirst=True
    )

    df["Book Returned"] = pd.to_datetime(
        df["Book Returned"],
        errors="coerce",
        dayfirst=True
    )

    df = data_enrich(df, "Book checkout", "Book Returned")

    return df


def clean_customers_data(df):
    """Clean the customers dataframe and set correct data types."""
    df = remove_blank_rows(df)

    df["Customer ID"] = pd.to_numeric(
        df["Customer ID"],
        errors="coerce"
    ).astype("Int64")

    df["Customer Name"] = (
    df["Customer Name"]
    .astype("string")
    .str.strip()
    )

    return df


def split_valid_invalid_books(df):
    """
    Split books data into valid and invalid dataframes.

    Invalid if:
    - any column contains null
    - checkout date is missing/invalid
    - returned date is missing/invalid
    - checkout or returned date is before 2023 or after 2026
    """
    date_columns = ["Book checkout", "Book Returned"]

    null_mask = df.isna().any(axis=1)

    invalid_date_mask = pd.Series(False, index=df.index)

    for col in date_columns:
        invalid_date_mask = (
            invalid_date_mask
            | df[col].isna()
            | (df[col].dt.year < 2023)
            | (df[col].dt.year > 2026)
        )

    invalid_mask = null_mask | invalid_date_mask

    invalid_df = df[invalid_mask].copy()
    valid_df = df[~invalid_mask].copy()

    return valid_df, invalid_df


def split_valid_invalid_customers(df):
    """Split customers into valid and invalid dataframes based on null values."""
    invalid_mask = df.isna().any(axis=1)

    invalid_df = df[invalid_mask].copy()
    valid_df = df[~invalid_mask].copy()

    return valid_df, invalid_df


def save_to_csv(df, file_path):
    """Save dataframe to CSV without the index."""
    df.to_csv(file_path, index=False)
                                                                                                                                                        
                                        

In [6]:
# Load raw files
books_raw = file_loader("data/03_Library Systembook.csv") 
customers_raw = file_loader("data/03_Library SystemCustomers.csv")

# Clean files
books_clean = clean_books_data(books_raw) 
customers_clean = clean_customers_data(customers_raw)

# Split valid and invalid records
books_valid, books_invalid = split_valid_invalid_books(books_clean)
customers_valid, customers_invalid = split_valid_invalid_customers(customers_clean)

# Save outputs
save_to_csv(books_valid, "data/books_valid.csv") 
save_to_csv(books_invalid, "data/books_invalid.csv") 
save_to_csv(customers_valid, "data/customers_valid.csv")

# Only save customers_invalid if you want it # save_to_csv(customers_invalid, "data/customers_invalid.csv")

print("Books rows:", len(books_clean))
print("Books valid:", len(books_valid))
print("Books invalid:", len(books_invalid))

print("Customers rows:", len(customers_clean)) 
print("Customers valid:", len(customers_valid)) 
print("Customers invalid:", len(customers_invalid))

display(books_valid)
display(books_invalid)
display(customers_valid)


Books rows: 21
Books valid: 18
Books invalid: 3
Customers rows: 8
Customers valid: 8
Customers invalid: 0


,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID,days_borrowed
0,1,Catcher in the Rye,2023-02-20,2023-02-25,2 weeks,1,5.0
1,2,Lord of the rings the two towers,2023-03-24,2023-03-21,2 weeks,2,-3.0
2,3,Lord of the rings the return of the kind,2023-03-29,2023-03-25,2 weeks,3,-4.0
3,4,The hobbit,2023-04-02,2023-03-25,2 weeks,4,-8.0
4,5,Dune,2023-04-02,2023-03-25,2 weeks,5,-8.0
5,6,Little Women,2023-04-02,2023-05-01,2 weeks,1,29.0
7,8,Misery,2023-04-15,2023-04-03,2 weeks,7,-12.0
8,9,Catch 22,2023-04-15,2023-04-16,2 weeks,7,1.0
9,10,Animal Farm,2023-04-20,2023-04-24,2 weeks,2,4.0
10,11,1984,2023-04-23,2023-04-27,2 weeks,8,4.0


,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID,days_borrowed
6,7,IT,2063-04-10,2023-04-03,2 weeks,6,-14617.0
16,17,The Bloody Chamber,NaT,2023-06-04,2 weeks,3,NaN
20,21,<NA>,2023-06-01,2023-06-05,2 weeks,<NA>,4.0


,Customer ID,Customer Name
0,1,Jane Doe
1,2,John Smith
2,3,Dan Reeves
4,5,William Holden
5,6,Jaztyn Forest
6,7,Jackie Irving
7,8,Matthew Stirling
8,9,Emory Ted
